First version. Only valid ISBNs obtained by removing invalid characters (but result not checked for actual existence). Only non-zero ratings. Only users and books that have at least 3 ratings.

Update: U and V generated using wide uniform distribution

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix

from load_data import load_data
from valid_test_select import valid_test_select
from initialize_model import initialize_mu_b_c
from helpers import get_rmse
from fit_model import fit_model_no_UV, fit_model_full

In [3]:
np.set_printoptions(linewidth=150, precision=6)  # 75, 8

# Loading Data

In [4]:
df = load_data()

In [5]:
df.agg(["min", "max", "nunique", "dtype", "count", "size"])

,userid,isbn,rating
min,2,0000001481,0
max,278854,9998150752,10
nunique,103371,328465,11
dtype,int32,object,int8
count,1135338,1135338,1135338
size,1135338,1135338,1135338


In [6]:
df = df[df.rating > 0]

In [7]:
df.agg(["min", "max", "nunique", "size"])

,userid,isbn,rating
min,8,000003827X,1
max,278854,9997555635,10
nunique,76331,179924,10
size,427261,427261,427261


# Creating Matrix Y

In [8]:
# Encoding userids and isbns as categories (integers 0 to N-1)

user_cats = df.userid.astype("category")
book_cats = df.isbn.astype("category")

In [9]:
# Creating sparse matrix and converting it to pd.DataFrame

Y_sparse = coo_matrix((df.rating.values, (user_cats.cat.codes, book_cats.cat.codes)))
Y = pd.DataFrame(Y_sparse.toarray())

In [10]:
# print("Y shape:", Y.shape)  # (76331, 179924)
# print("total entries:", (Y > 0).sum().sum())  # 427261
# print("avg number of ratings per user:", round((Y > 0).sum(axis=1).mean(), 2))  # 5.6
# print("avg number of ratings per book:", round((Y > 0).sum(axis=0).mean(), 2))  # 2.37

In [11]:
# Filtering out improbably active readers

# np.sort(np.partition((Y > 0).sum(axis=1), -20)[-20:])
Y = Y.loc[(Y > 0).sum(axis=1) < 2000, :]

In [12]:
# Filtering users and books with enough observations

min_rats = 3
old_rows, old_cols = Y.shape
while True:
    Y_pos = Y > 0  # type: ignore
    user_rats = Y_pos.sum(axis=1)  # type: ignore
    book_rats = Y_pos.sum(axis=0)  # type: ignore
    new_rows = (user_rats >= min_rats).sum()
    new_cols = (book_rats >= min_rats).sum()
    # print(new_rows, new_cols)
    Y = Y.loc[user_rats >= min_rats, book_rats >= min_rats]  # type: ignore
    if (old_rows == new_rows) and (old_cols == new_cols):
        break
    old_rows, old_cols = new_rows, new_cols

In [13]:
print("Y shape:", Y.shape)
print("total entries:", (Y > 0).sum().sum())
print("avg number of ratings per user:", round((Y > 0).sum(axis=1).mean(), 2))
print("avg number of ratings per book:", round((Y > 0).sum(axis=0).mean(), 2))

Y shape: (14903, 22661)
total entries: 182612
avg number of ratings per user: 12.25
avg number of ratings per book: 8.06


In [14]:
# Converting Y to numpy array

Y_columns = Y.columns.copy()
Y = Y.to_numpy()

In [15]:
def get_isbn(pos):
    """Given position of column in matrix Y, return the original ISBN."""
    return book_cats.cat.categories[Y_columns[pos]]

# Selecting validation and test sets

In [16]:
Y_train, valid_data, test_data = valid_test_select(Y, 20_000, 20_000, seed=123)

In [17]:
print("Y_train shape:", Y_train.shape)
print("total train entries:", (Y_train > 0).sum().sum())
print("avg. # of train entries per user:", round((Y_train > 0).sum(axis=1).mean(), 2))
print("avg. # of train entries per book:", round((Y_train > 0).sum(axis=0).mean(), 2))
print("users with no ratings:", ((Y_train > 0).sum(axis=1) == 0).sum())
print("books with no ratings:", ((Y_train > 0).sum(axis=0) == 0).sum())

Y_train shape: (14903, 22661)
total train entries: 142612
avg. # of train entries per user: 9.57
avg. # of train entries per book: 6.29
users with no ratings: 50
books with no ratings: 103


# ALS

In [18]:
Y_csr = csr_matrix(Y_train)
Y_csc = Y_csr.tocsc()

## Model with just a training global mean

In [19]:
mu, _, _ = initialize_mu_b_c(Y_train)
print("test rmse of model=mu:", get_rmse(mu, test_data))
# test rmse of model=mu: 1.8087624016357353

test rmse of model=mu: 1.7927095878715544


## Model with only biases

In [20]:
lambda_bias = 4.5
mu, b, c = initialize_mu_b_c(Y_train)
mu, b, c, valid_rmse = fit_model_no_UV(
    Y_csr, Y_csc, mu, b, c, lambda_bias, valid_data, 1e-5, info=2
)

original valid rmse: 1.7860372964922988
counter=1, max_norm_diff=111.8991713866579, valid_rmse=1.5794323601860292, valid_rmse_diff=0.20660493630626964
counter=2, max_norm_diff=16.108281174398396, valid_rmse=1.5644711116060812, valid_rmse_diff=0.014961248579947961
counter=3, max_norm_diff=2.2115732036853477, valid_rmse=1.5634312022543255, valid_rmse_diff=0.0010399093517556857
counter=4, max_norm_diff=1.079842766197419, valid_rmse=1.5630778316780685, valid_rmse_diff=0.0003533705762570616
counter=5, max_norm_diff=0.7986767524302414, valid_rmse=1.5628368481666877, valid_rmse_diff=0.000240983511380799
counter=6, max_norm_diff=0.6279593267796958, valid_rmse=1.562651124666763, valid_rmse_diff=0.00018572349992473924
counter=7, max_norm_diff=0.5432874637838129, valid_rmse=1.5625071497054592, valid_rmse_diff=0.00014397496130369447
counter=8, max_norm_diff=0.4570386727169688, valid_rmse=1.5623962827293951, valid_rmse_diff=0.00011086697606410567
counter=9, max_norm_diff=0.37696025350172585, valid_

In [21]:
test_preds = np.clip(mu + b[test_data.rows] + c[test_data.cols], 1, 10)
test_rmse = get_rmse(test_preds, test_data)
print("test rmse of a tuned model with only biases:", test_rmse)
# test rmse of a tuned model with only biases: 1.5704328421158182

test rmse of a tuned model with only biases: 1.569357938911514


## Full model

### k=1

In [46]:
k = 1
lambda_bias = 4.5
lambda_fact = 24
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.uniform(-20, 20, size=(Y.shape[0], k))
V = rng.uniform(-20, 20, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 5.200346792260443
counter=1, max_norm_diff=4196.097502560533, valid_rmse=5.004227552828167, valid_rmse_diff=0.19611923943227616
counter=2, max_norm_diff=3628.36043999737, valid_rmse=3.7598827200678535, valid_rmse_diff=1.244344832760313
counter=3, max_norm_diff=776.9561399610296, valid_rmse=2.062813273336481, valid_rmse_diff=1.6970694467313727
counter=4, max_norm_diff=170.03830631254138, valid_rmse=1.6074476556137247, valid_rmse_diff=0.4553656177227561
counter=5, max_norm_diff=36.689986638881535, valid_rmse=1.5656502620985917, valid_rmse_diff=0.04179739351513301
counter=6, max_norm_diff=8.114800127368282, valid_rmse=1.5632452925234304, valid_rmse_diff=0.002404969575161342
counter=7, max_norm_diff=1.8876970972432068, valid_rmse=1.5628344714446154, valid_rmse_diff=0.0004108210788149602
counter=8, max_norm_diff=1.462411370457636, valid_rmse=1.5625971197543558, valid_rmse_diff=0.000237351690259624
counter=9, max_norm_diff=1.281137005863865, valid_rmse=1.5624635135532539

In [23]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.5704970647787702

test rmse of full model: 1.5697468577018228


### k=2

In [24]:
k = 2
lambda_bias = 4.5
lambda_fact = 18.5
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.uniform(-20, 20, size=(Y.shape[0], k))
V = rng.uniform(-20, 20, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 5.26842318122828
counter=1, max_norm_diff=5980.984765539763, valid_rmse=5.136944531987756, valid_rmse_diff=0.131478649240524
counter=2, max_norm_diff=4866.252770195093, valid_rmse=4.398546405905988, valid_rmse_diff=0.7383981260817674
counter=3, max_norm_diff=1217.745830101025, valid_rmse=2.834022965346443, valid_rmse_diff=1.5645234405595456
counter=4, max_norm_diff=352.6216291781646, valid_rmse=1.8720837441730722, valid_rmse_diff=0.9619392211733706
counter=5, max_norm_diff=85.49647882896407, valid_rmse=1.6345849799680587, valid_rmse_diff=0.23749876420501348
counter=6, max_norm_diff=22.55733188325376, valid_rmse=1.587249210441995, valid_rmse_diff=0.047335769526063665
counter=7, max_norm_diff=8.19903583047565, valid_rmse=1.573517320513076, valid_rmse_diff=0.013731889928919161
counter=8, max_norm_diff=4.399107554092891, valid_rmse=1.5667381126237168, valid_rmse_diff=0.006779207889359151
counter=9, max_norm_diff=2.9115678748342027, valid_rmse=1.5644062657522302, valid_

In [25]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.5717291565825728

test rmse of full model: 1.5710564409801586


### k=4

In [26]:
k = 4
lambda_bias = 4.6
lambda_fact = 25
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.uniform(-20, 20, size=(Y.shape[0], k))
V = rng.uniform(-20, 20, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 5.328378495002235
counter=1, max_norm_diff=8327.671562612119, valid_rmse=5.264240101841594, valid_rmse_diff=0.06413839316064074
counter=2, max_norm_diff=6506.588552270267, valid_rmse=4.843696809279218, valid_rmse_diff=0.42054329256237555
counter=3, max_norm_diff=1828.4429638439728, valid_rmse=3.6896992282602343, valid_rmse_diff=1.153997581018984
counter=4, max_norm_diff=643.7440882652305, valid_rmse=2.3579880132594657, valid_rmse_diff=1.3317112150007686
counter=5, max_norm_diff=199.04066578392656, valid_rmse=1.787408797646178, valid_rmse_diff=0.5705792156132876
counter=6, max_norm_diff=60.470250815720114, valid_rmse=1.6190008342707372, valid_rmse_diff=0.16840796337544095
counter=7, max_norm_diff=20.65640092816946, valid_rmse=1.57419137613609, valid_rmse_diff=0.044809458134647207
counter=8, max_norm_diff=9.283560850578672, valid_rmse=1.564524321795132, valid_rmse_diff=0.009667054340958048
counter=9, max_norm_diff=5.008402018255848, valid_rmse=1.5636605859219113, val

In [27]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.5703045287839543

test rmse of full model: 1.5693454981767465


### k=8

In [28]:
k = 8
lambda_bias = 4.4
lambda_fact = 20
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.uniform(-20, 20, size=(Y.shape[0], k))
V = rng.uniform(-20, 20, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 5.286425315355606
counter=1, max_norm_diff=12084.523609148922, valid_rmse=5.180715772492115, valid_rmse_diff=0.1057095428634911
counter=2, max_norm_diff=8316.54377761616, valid_rmse=4.999188317768289, valid_rmse_diff=0.1815274547238257
counter=3, max_norm_diff=2829.367136137436, valid_rmse=4.531021859949177, valid_rmse_diff=0.4681664578191125
counter=4, max_norm_diff=1223.9754200946968, valid_rmse=3.6911469764806855, valid_rmse_diff=0.8398748834684913
counter=5, max_norm_diff=590.742536072497, valid_rmse=2.813141221769596, valid_rmse_diff=0.8780057547110895
counter=6, max_norm_diff=283.03522983290924, valid_rmse=2.2067220179475986, valid_rmse_diff=0.6064192038219973
counter=7, max_norm_diff=129.92892822389055, valid_rmse=1.8673176516977366, valid_rmse_diff=0.33940436624986203
counter=8, max_norm_diff=57.64667373190636, valid_rmse=1.7136171616535478, valid_rmse_diff=0.15370049004418873
counter=9, max_norm_diff=33.3137730579441, valid_rmse=1.629615376266441, valid_rm

In [29]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.5702971262712437

test rmse of full model: 1.5691587253662032


### k=16

In [30]:
k = 16
lambda_bias = 4.3
lambda_fact = 14.7
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.uniform(-20, 20, size=(Y.shape[0], k))
V = rng.uniform(-20, 20, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 5.345172413957367
counter=1, max_norm_diff=17430.70306394037, valid_rmse=5.254492310073458, valid_rmse_diff=0.09068010388390846
counter=2, max_norm_diff=10511.998499738798, valid_rmse=5.158187602111118, valid_rmse_diff=0.09630470796234025
counter=3, max_norm_diff=4333.0093277209535, valid_rmse=5.009912677712762, valid_rmse_diff=0.14827492439835588
counter=4, max_norm_diff=1931.5198729163121, valid_rmse=4.737029804201116, valid_rmse_diff=0.2728828735116462
counter=5, max_norm_diff=988.5325735782287, valid_rmse=4.330513550514643, valid_rmse_diff=0.4065162536864735
counter=6, max_norm_diff=615.8771256305256, valid_rmse=3.8248553744014293, valid_rmse_diff=0.5056581761132133
counter=7, max_norm_diff=410.31853895476394, valid_rmse=3.330514171455027, valid_rmse_diff=0.49434120294640227
counter=8, max_norm_diff=281.22024627285634, valid_rmse=2.9049718827206035, valid_rmse_diff=0.42554228873442357
counter=9, max_norm_diff=192.81038254195488, valid_rmse=2.549717593577501, va

In [31]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.5717814048614096

test rmse of full model: 1.5703207806790256


### k=32

In [39]:
k = 32
lambda_bias = 4.3
lambda_fact = 13.7
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.uniform(-20, 20, size=(Y.shape[0], k))
V = rng.uniform(-20, 20, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 5.337447834544822
counter=1, max_norm_diff=24650.981228178964, valid_rmse=5.320372901788236, valid_rmse_diff=0.01707493275658578
counter=2, max_norm_diff=14090.656731221077, valid_rmse=5.255172633018755, valid_rmse_diff=0.06520026876948126
counter=3, max_norm_diff=6077.296198599668, valid_rmse=5.173133376992354, valid_rmse_diff=0.08203925602640094
counter=4, max_norm_diff=2916.016825454374, valid_rmse=5.035764407050613, valid_rmse_diff=0.13736896994174064
counter=5, max_norm_diff=1529.6968034811475, valid_rmse=4.849850860972617, valid_rmse_diff=0.18591354607799637
counter=6, max_norm_diff=880.6693697897409, valid_rmse=4.611443732550428, valid_rmse_diff=0.23840712842218892
counter=7, max_norm_diff=561.100232536628, valid_rmse=4.340471074276901, valid_rmse_diff=0.2709726582735268
counter=8, max_norm_diff=407.03052042584426, valid_rmse=4.067656897079272, valid_rmse_diff=0.27281417719762935
counter=9, max_norm_diff=310.873188919184, valid_rmse=3.8011889677134163, valid

In [41]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.5699819711228356

test rmse of full model: 1.5696710384371007


### k=256

In [ ]:
k = 256
lambda_bias = 4
lambda_fact = 10
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.uniform(-20, 20, size=(Y.shape[0], k))
V = rng.uniform(-20, 20, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

In [35]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.568533842304493

test rmse of full model: 1.5716560890330094


### k=512

In [36]:
k = 512
lambda_bias = 3.9
lambda_fact = 10
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.uniform(-20, 20, size=(Y.shape[0], k))
V = rng.uniform(-20, 20, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 5.351797414533016
counter=1, max_norm_diff=103753.03410204069, valid_rmse=5.313275834547766, valid_rmse_diff=0.0385215799852503
counter=2, max_norm_diff=55701.10674297039, valid_rmse=5.299997366789243, valid_rmse_diff=0.01327846775852315
counter=3, max_norm_diff=25057.79421235973, valid_rmse=5.316870949300243, valid_rmse_diff=-0.016873582510999796
valid_rmse_diff is negative, returning the last iteration's values
results: counter=2, max_norm_diff=55701.10674297039, valid_rmse=5.316870949300243, valid_rmse_diff=0.01327846775852315


In [37]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.5690911174883715

test rmse of full model: 5.3084382077888


In [38]:
# seed = 103
# global mean                           test_rmse = 1.8087624016357353
# only biases                           test_rmse = 1.5704328421158182
# k = 1,   l_bias = 4.5, l_fact = 24,   test_rmse = 1.5704970647787702
# k = 2,   l_bias = 4.5, l_fact = 18.5, test_rmse = 1.5717291565825728
# k = 4,   l_bias = 4.6, l_fact = 25,   test_rmse = 1.5703045287839543
# k = 8,   l_bias = 4.4, l_fact = 20,   test_rmse = 1.5702971262712437
# k = 16,  l_bias = 4.3, l_fact = 14.7, test_rmse = 1.5717814048614096
# k = 32,  l_bias = 4.3, l_fact = 13.7, test_rmse = 1.5699819711228356
# k = 256, l_bias = 4,   l_fact = 10    test_rmse = 1.568533842304493
# k = 512, l_bias = 3.9, l_fact = 10,   test_rmse = 1.5690911174883715